In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# Normalize the images
train_images = train_images / 255.0
test_images = test_images / 255.0

# Class names for Fashion MNIST
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Reshape the data for CNN input: add channel dimension
img_rows_fashion, img_cols_fashion = 28, 28
train_images = train_images.reshape(train_images.shape[0], img_rows_fashion, img_cols_fashion, 1)
test_images = test_images.reshape(test_images.shape[0], img_rows_fashion, img_cols_fashion, 1)
input_shape_fashion = (img_rows_fashion, img_cols_fashion, 1)

print(f'New training set shape for Fashion MNIST: {train_images.shape}')
print(f'New test set shape for Fashion MNIST: {test_images.shape}')

# Number of classes
num_classes_fashion = len(class_names)

# One-hot encode the labels
y_train_fashion = to_categorical(train_labels, num_classes_fashion)
y_test_fashion = to_categorical(test_labels, num_classes_fashion)

print(f'y_train_fashion shape: {y_train_fashion.shape}')
print(f'y_test_fashion shape: {y_test_fashion.shape}')

# Define the convolutional model for Fashion MNIST
model_fashion = Sequential([
    Conv2D(64, kernel_size=(3, 3), activation='relu', input_shape=input_shape_fashion),
    BatchNormalization(),
    Conv2D(64, kernel_size=(3, 3), activation='relu'), # Added another Conv2D layer
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25), # Placed dropout earlier
    Conv2D(128, kernel_size=(3, 3), activation='relu'),
    BatchNormalization(),
    Conv2D(128, kernel_size=(3, 3), activation='relu'), # Added another Conv2D layer
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25), # Adjusted dropout position
    Flatten(),
    Dense(128, activation='relu'), # Increased Dense layer size
    Dropout(0.5),
    Dense(num_classes_fashion, activation='softmax')
])

# Display the model's architecture
model_fashion.summary()

# Compile the model
model_fashion.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(),
    metrics=['accuracy']
)

In [ ]:
# Training the model
history_fashion = model_fashion.fit(
    train_images,
    y_train_fashion,
    epochs=15, # Using 15 epochs as in the MNIST example
    batch_size=128,
    validation_split=0.2
)


In [ ]:
# Plot the training history
plt.figure(figsize=(12, 8))

# Plot training & validation accuracy values
plt.subplot(2, 1, 1)
plt.plot(history_fashion.history['accuracy'], label='Train Accuracy')
plt.plot(history_fashion.history['val_accuracy'], label='Validation Accuracy')
plt.title('Fashion MNIST Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='upper left')

# Plot training & validation loss values
plt.subplot(2, 1, 2)
plt.plot(history_fashion.history['loss'], label='Train Loss')
plt.plot(history_fashion.history['val_loss'], label='Validation Loss')
plt.title('Fashion MNIST Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper left')

plt.tight_layout()
plt.show()

# Evaluate the model's performance on the training data
train_loss_fashion, train_accuracy_fashion = model_fashion.evaluate(train_images, y_train_fashion, verbose=0)
print(f'Fashion MNIST Train accuracy: {train_accuracy_fashion:.4f}, Train loss: {train_loss_fashion:.4f}')

# Evaluate the model's performance on the test data
test_loss_fashion, test_accuracy_fashion = model_fashion.evaluate(test_images, y_test_fashion, verbose=0)
print(f'Fashion MNIST Test accuracy: {test_accuracy_fashion:.4f}, Test loss: {test_loss_fashion:.4f}')

# Predict probabilities on test data
predictions_fashion = model_fashion.predict(test_images)

# Convert predicted probabilities to class labels
predicted_labels_fashion = np.argmax(predictions_fashion, axis=1)

# Print the first few predicted digits to verify
print("First 10 predicted labels for Fashion MNIST:", predicted_labels_fashion[:10])
print("First 10 true labels for Fashion MNIST:", test_labels[:10])

# Plot the first 25 images with their predicted and actual labels
plt.figure(figsize=(12, 12)) # Adjusted figure size for better visibility
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)

    # Get the image and true label
    img_display = test_images[i].reshape(img_rows_fashion, img_cols_fashion)
    true_label_idx = test_labels[i] # Original integer label
    predicted_label_idx = predicted_labels_fashion[i]

    # Set the title color based on the prediction correctness
    color_fashion = '#008800' if predicted_label_idx == true_label_idx else '#bb0000'

    # Plot the image
    plt.imshow(img_display, cmap=plt.cm.binary)
    plt.title(f'{class_names[predicted_label_idx]} ({class_names[true_label_idx]})', color=color_fashion, fontsize=8) # Using class names
plt.tight_layout()
plt.show()

# Create a confusion matrix
conf_matrix_fashion = confusion_matrix(test_labels, predicted_labels_fashion)

# Plot the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix_fashion, annot=True, fmt="d", cmap="YlGnBu", cbar=False,
            xticklabels=class_names, yticklabels=class_names, linewidths=.5) # Using class names for labels
plt.title('Fashion MNIST Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()
